# Mutiple Regression Using Meteostat Weather Data

In this notebook, we will build a linear regression model of daily wind speed for certain locations in Colorado. We will try to predict wind speed from temperature, pressure and relative humidity in each location. We will assess how good our model is by considering the r-squared value as well as residual and other plots.

To summarize our proposed model, here are the inputs and outputs.

Inputs (independent variables, represented by `X`):
- Temperature (deg C)
- Pressure (mbar)
- Relative Humidity (%)

Output (dependent variables, represented by `y` (actual) and `y_hat` (model):
- Wind speed (kmh)

You may want to refer to the [Meteostat Python Library Guide](https://dev.meteostat.net/python) for notes on using this library.

We will start by importing data from meteostat into a `pandas` dataframe. We'll collect data for 3 locations, but will start by modelling the wind speed for Alamosa.

In [ ]:
from datetime import date
import meteostat as ms, numpy as np, pandas as pd

# Set display options for numpy
np.set_printoptions(floatmode='fixed', precision=3)

# Set start and end date for data
start = date(2025, 1, 1)
end = date(2025, 12, 31)

# Select Station(s)
denver_stn = ms.stations.meta(72469)
alamosa_stn = ms.stations.meta(72462)
lamar_stn = ms.stations.meta('KLAA0')

# Get data for three stations in CO
denver_data = ms.daily(denver_stn, start, end)
alamosa_data = ms.daily(alamosa_stn, start, end)
lamar_data = ms.daily(lamar_stn, start, end)

In [ ]:
# Store Alamosa data in pandas dataframe and perform some data processing
alamosa = alamosa_data.fetch()
alamosa['pres'] = alamosa['pres'].interpolate()
alamosa = alamosa.reset_index()
alamosa['time'] = pd.to_datetime(alamosa['time'])
alamosa

## Part 1: Scatter Plots

A scatter plot allows you to plot many pairs of (x, y) points on the same axes. This gives you a way to see what relationship there may be between x and y values. We will create a scatter plot of wind speed against each of our independent variables to see if there is any apparent relationship between the variables. We will also calculate the correlation coefficient in each case.

In [ ]:
# Create a scatter plot of windspeed vs air pressure for Alamosa
import matplotlib.pyplot as plt
from scipy import stats

x1 = np.array(alamosa['wspd'])
y1 = np.array(alamosa['pres'])
plt.xlabel('wind speed (km/h)')
plt.ylabel('Air Pressure (mb)')
plt.title('Daily Avg Air Pressure vs Wind Speed in Alamosa, 2025')
plt.scatter(x1, y1)
plt.show()

m, b, r, p, std_err = stats.linregress(x1, y1)
print(f"slope (m): {m:.3f}")
print(f"intercept (b): {b:.3f}")
print(f"Correlation coefficient (r): {r:.3f}")
print(f"Standard error: {std_err:.3f}")

In [ ]:
# Create a scatter plot of windspeed vs temperature for Alamosa
x1 = alamosa['wspd']
y1 = alamosa['temp']
plt.xlabel('wind speed (km/h)')
plt.ylabel('temp (deg C)')
plt.title('Daily Avg Temperature vs Wind Speed in Alamosa, 2025')
plt.scatter(x1, y1)
plt.show()

m, b, r, p, std_err = stats.linregress(x1, y1)
print(f"slope (m): {m:.3f}")
print(f"intercept (b): {b:.3f}")
print(f"Correlation coefficient (r): {r:.3f}")
print(f"Standard error: {std_err:.3f}")

In [ ]:
# Create a scatter plot of windspeed vs relative humidity for Alamosa
x1 = alamosa['wspd']
y1 = alamosa['rhum']
plt.xlabel('wind speed (km/h)')
plt.ylabel('Rel. Humidity (deg C)')
plt.title('Daily Avg Rel Hum vs Wind Speed in Alamosa, 2025')
plt.scatter(x1, y1)
plt.show()

m, b, r, p, std_err = stats.linregress(x1, y1)
print(f"slope (m): {m:.3f}")
print(f"intercept (b): {b:.3f}")
print(f"Correlation coefficient (r): {r:.3f}")
print(f"Standard error: {std_err:.3f}")

## Checkpoint 1:
<div class="alert alert-success" role="alert"> 
 <span style="color:black"> 
Looking at the scatter plots and coefficients for wind speed versus the three independent variables. Which if any appear to have some linear association? Which is the strongest and which is the weakest? Is the association positive (both go up together) or negative (as one goes up the other goes down)? 
 </span>

**Write a couple of sentences here giving your answer.**

# Part 2: Create Regression Model

All three quantities (temperature, pressure, relative humidity) show some correlation with wind speed, so at this stage, let's include them all in our model.

Inputs (`X`):
- Air Pressure
- Temperature
- Relative Humidity

Output (`y`):
- Wind Speed

Now we will use the `LinearRegression` library from the `sklearn` package. This is a good library for multiple regression models.

In [ ]:
# Build model using sklearn
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

# Define independent variables (X) and the dependent variable (y)
X = alamosa[['temp', 'pres', 'rhum']] # Independent variables
y = alamosa['wspd']  # Single dependent variable


# Create and train the model
model = LinearRegression()
model.fit(X, y)

# Make predictions and add them as a column in the dataframe
alamosa['Pred Wspd'] = model.predict(X)

# Print the coefficients and intercept
print(f"Coefficients: {model.coef_}")
print(f"Intercept: {model.intercept_}")

# Evaluate the model (e.g., using r-squared score)
r_sq = model.score(X, y)
print(f"R-squared score: {r_sq:.4f}")

In [ ]:
# Plot wind speed and predicted wind speed over time to compare
plt.plot(alamosa['time'], alamosa['wspd'], label='Actual')
plt.plot(alamosa['time'], alamosa['Pred Wspd'], label='Model')
plt.legend()
plt.show()

In [ ]:
# Create and plot residuals
residuals = alamosa['wspd'] - alamosa['Pred Wspd']
x = np.arange(0, len(residuals))

# Plot residuals
plt.scatter(x, residuals)
plt.plot([x.min(), x.max()], [0,0])
for i in range(len(residuals)):
    plt.plot([x[i], x[i]], [0, residuals[i]], ':b')
plt.title("Residual Plot for Model")
plt.xlabel("Day of Year (2025)")
plt.ylabel("Residual: y - y_hat")
plt.show()

## Checkpoint 2: 
<div class="alert alert-success" role="alert"> 
 <span style="color:black"> 
What are your observations? How good is the model? Is a linear model a good fit? Are some of the inputs better predictors that others? Could we drop any of the inputs from the model without losing too much accuracy? Write your answers below.
 </span>

**Write your answers here.**

## Task 1

Pick one of the other locations provided (or choose a different location if you wish) and use the data from Meteostat to construct a multiple regression model. Compare your model predictions to the actual values. How good is your model? How does your model compare to the model for Alamosa that we created above?

In [ ]:
# Type your code here


## Task 2

Using one of the models already created in this notebook, use Meteostat to collect daily weather data for the same location for a **different year**. The use the model already created to predict daily wind speeds for the year you have chosen. How good is your model at predicting wind speed for the year that you have chosen?

In [ ]:
# Type your code here
